# Task 4 - Game Title Detection from Screenshots (5 classes, metric: F1 macro)

5-fold ConvNeXt-Small + EfficientNetV2-S ensemble at 192x384: EMA weights, warmup + cosine LR, head LR x5, early stopping on the best EMA epoch, flip + multi-scale TTA, OOF stacking. Only `train` is used for training and model selection.

- **Data pipeline:** JPEG decoding in C++ (`decode_jpeg`), uint8 batches moved to the GPU, resize / normalise / random erasing on the GPU; `torch.compile`, bf16, TF32, `channels_last`.
- **Augmentation (`AUG`):** `strong` (default) adds random-resized-crop (60-100 %), TrivialAugmentWide, blur and grayscale. Test screenshots come from different sources than train (a train-vs-test classifier on thumbnails reaches AUC 0.76 and class shares differ strongly), so the model must not rely on source-specific framing or colour. `light` reproduces the reference run.
- **Resume:** every finished fold is cached (file name includes `AUG` and image size) and reloaded on a re-run.
- **`PRIOR_EM` (off):** optional EM label-shift correction on the unlabeled test probabilities; it uses test data, so confirm with the TA before enabling it.
- **Outputs:** `task4_pred.csv` (id, pred, p0-p4), `task4_only_submission.csv`, `task4_oof_proba.npy`, `task4_test_proba.npy`.

Reference run (`AUG=light`, 192x384, L4 GPU): OOF macro-F1 ConvNeXt-S 0.98949, EfficientNetV2-S 0.98456, mean blend **0.99056** (stacking 0.99001, not used).

Runs on Colab (GPU runtime, data from Google Drive) or locally from the repo layout (`task4/train`, `task4/test`, `task4/train.csv`, `task4/test.csv`). `img_size = (128, 256)` is about 2x faster.

## 1. Environment and data

In [ ]:
import zipfile, os, glob, shutil, subprocess, sys, torch
from pathlib import Path
try:
    from google.colab import drive, auth
    IN_COLAB = True
except ImportError:      # running from the repo (task4/train, task4/test, task4/train.csv, task4/test.csv)
    IN_COLAB = False

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Colab: Runtime -> Change runtime type -> GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) | BF16: {torch.cuda.is_bf16_supported()} | CPUs: {os.cpu_count()}")

# Quick CUDA kernel sanity check
_a = torch.randn(8, 8, device="cuda")
_ = _a @ _a
torch.cuda.synchronize()
print("CUDA kernel check passed")

# Ensure timm is installed
try:
    import timm
    print("timm:", timm.__version__)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm"])
    import timm
    print("Installed timm:", timm.__version__)

if not IN_COLAB:
    _root = Path.cwd()
    DATA_DIR = str(_root / "task4" if (_root / "task4" / "train.csv").exists() else _root)
    print(f"DATA_DIR = '{DATA_DIR}'")
else:
    drive.mount("/content/drive")

    # extract to the RAM disk when it has >= 3 GB free (no disk I/O during training)
    shm_free_gb = shutil.disk_usage("/dev/shm").free / (1024 ** 3) if os.path.exists("/dev/shm") else 0.0
    if shm_free_gb >= 3.0:
        EXTRACT_DIR = "/dev/shm/data"
        print(f"RAM disk: {EXTRACT_DIR} ({shm_free_gb:.1f} GB free)")
    else:
        EXTRACT_DIR = "/content/data"
        print(f"local disk: {EXTRACT_DIR} (/dev/shm has {shm_free_gb:.1f} GB free)")
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    zip_files = sorted(glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True))
    task4_zips = [z for z in zip_files if "task4" in os.path.basename(z).lower()]
    LOCAL_ZIP = "/content/task4_image_classification.zip"
    GDRIVE_FILE_ID = "1C8csylbBo3Oab24ODGKPVom3_SDUJ_Qw"

    if not glob.glob(f"{EXTRACT_DIR}/**/train.csv", recursive=True):
        if task4_zips:
            ZIP_PATH = task4_zips[0]
            print(f"zip found in Drive: {ZIP_PATH}")
        else:
            if not os.path.exists(LOCAL_ZIP):
                print("task4*.zip not found in MyDrive, downloading by file id")
                ok = False
                try:
                    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
                    subprocess.check_call(["gdown", GDRIVE_FILE_ID, "-O", LOCAL_ZIP])
                    ok = os.path.exists(LOCAL_ZIP) and os.path.getsize(LOCAL_ZIP) > 1_000_000
                except Exception:
                    ok = False
                if not ok:
                    print("gdown failed, downloading through the Drive API")
                    auth.authenticate_user()
                    from googleapiclient.discovery import build
                    from googleapiclient.http import MediaIoBaseDownload
                    svc = build("drive", "v3")
                    req = svc.files().get_media(fileId=GDRIVE_FILE_ID)
                    with open(LOCAL_ZIP, "wb") as fh:
                        dl = MediaIoBaseDownload(fh, req, chunksize=50 * 1024 * 1024)
                        done_dl = False
                        while not done_dl:
                            status, done_dl = dl.next_chunk()
                            if status:
                                print(f"   Downloading... {int(status.progress() * 100)}%")
            ZIP_PATH = LOCAL_ZIP

        print(f"extracting to {EXTRACT_DIR}")
        with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
            members = [m for m in zip_ref.namelist() if not m.startswith("__MACOSX/")]
            zip_ref.extractall(EXTRACT_DIR, members=members)
        print(f"extracted {len(members):,} files")
    else:
        print(f"data already in {EXTRACT_DIR}")

    train_csv_paths = glob.glob(f"{EXTRACT_DIR}/**/train.csv", recursive=True)
    assert len(train_csv_paths) > 0, "train.csv not found in the zip"
    DATA_DIR = os.path.dirname(train_csv_paths[0])
    print(f"DATA_DIR = '{DATA_DIR}'")


## 2. Configuration and data pipeline

In [ ]:
import os, gc, random, time, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.io import read_file, decode_jpeg, ImageReadMode
from torchvision.transforms import v2
import timm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

SEED = 42
DATA = Path(DATA_DIR)
SAVE_DIR = Path("/content/drive/MyDrive/CPE342_Task4_478e13c") if IN_COLAB else Path(DATA_DIR) / "outputs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
(SAVE_DIR / "models").mkdir(parents=True, exist_ok=True)
RESUME = True   # reload finished folds from SAVE_DIR instead of retraining

AUG = os.environ.get("AUG", "strong")          # "strong" | "light" (reference run)
PRIOR_EM = os.environ.get("PRIOR_EM") == "1"   # label-shift correction on test probabilities (off)
TASK = 4
NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
MODEL_LR = {
    "convnext_small.fb_in22k_ft_in1k": 2e-4,
    "convnext_base.fb_in22k_ft_in1k": 1.5e-4,
    "tf_efficientnetv2_s.in21k_ft_in1k": 6e-4,
}
MODELS = [
    "convnext_small.fb_in22k_ft_in1k",
    "tf_efficientnetv2_s.in21k_ft_in1k",
]
MODEL_NAME = "+".join(MODELS)

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
AUTO_WORKERS = min(10, max(2, (os.cpu_count() or 4) - 2))
AUTO_VAL_BS = 512 if VRAM_GB >= 70 else (256 if VRAM_GB >= 35 else 128)

CFG = {
    "n_folds": 5,
    "train_folds": [0, 1, 2, 3, 4],
    "native_size": (120, 230),        # stored image size; uint8 batches are resized on the GPU
    "img_size": (192, 384),           # (128, 256) is about 2x faster
    "batch_size": 64,
    "val_batch_size": AUTO_VAL_BS,
    "epochs": 12,
    "early_stop_patience": 3,         # epochs without a val macro-F1 gain > min_delta
    "early_stop_min_delta": 1e-4,
    "early_stop_min_epoch": 4,
    "use_compile": True,
    "head_lr_mult": 5.0,
    "weight_decay": 1e-2,
    "ema_decay": 0.998,
    "warmup_epochs": 1,
    "num_classes": 5,
    "num_workers": AUTO_WORKERS,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
print(
    f"Device: {CFG['device']} ({VRAM_GB:.1f} GB) | AMP: {'bf16' if USE_BF16 else 'fp16'} | "
    f"compile: {CFG['use_compile']} | img_size: {CFG['img_size']} | "
    f"val_bs: {CFG['val_batch_size']} | workers: {CFG['num_workers']}"
)
print(f"Early Stopping: patience={CFG['early_stop_patience']} (min_delta={CFG['early_stop_min_delta']}, min_ep={CFG['early_stop_min_epoch']})")
print("SAVE_DIR:", SAVE_DIR)

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # TF32 + cuDNN autotune
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

seed_everything()

NORM = dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
MEAN_GPU = torch.tensor(NORM["mean"], device=CFG["device"], dtype=torch.float32).view(1, 3, 1, 1)
STD_GPU = torch.tensor(NORM["std"], device=CFG["device"], dtype=torch.float32).view(1, 3, 1, 1)

# CPU augmentations run on uint8 tensors at the native 120x230 size
light_cpu_transform = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    v2.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.05)),
])
# strong: robustness to the train/test source shift (framing, colour, sharpness)
strong_cpu_transform = v2.Compose([
    v2.RandomResizedCrop(CFG["native_size"], scale=(0.6, 1.0), ratio=(1.6, 2.3), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.TrivialAugmentWide(),
    v2.RandomGrayscale(p=0.05),
    v2.RandomApply([v2.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),
])
train_cpu_transform = strong_cpu_transform if AUG == "strong" else light_cpu_transform
print("augmentation:", AUG, "| PRIOR_EM:", PRIOR_EM)

def em_label_shift(P, prior_train, iters=200, tol=1e-7):
    """Saerens et al. (2002) EM: re-estimate class shares of an unlabeled set and re-weight its probabilities (no labels used)."""
    pi = prior_train.copy()
    for _ in range(iters):
        Q = P * (pi / prior_train); Q /= Q.sum(1, keepdims=True)
        new = Q.mean(0)
        if np.abs(new - pi).max() < tol:
            break
        pi = new
    return Q, pi

def gpu_preprocess(x_u8, erasing=False):
    """uint8 (B,3,120,230) -> GPU float, resize to img_size, normalise, optional random erasing."""
    x = x_u8.to(CFG["device"], dtype=torch.float32, non_blocking=True)
    if x.shape[-2:] != CFG["img_size"]:
        x = F.interpolate(x, size=CFG["img_size"], mode="bilinear", align_corners=False, antialias=True)
    x = x.mul_(1.0 / 255.0).sub_(MEAN_GPU).div_(STD_GPU)
    if erasing:
        # per-sample random erasing without a CPU-GPU sync
        B, _, H, W = x.shape
        area = H * W
        for idx in range(B):
            if random.random() < 0.25:
                target_area = area * random.uniform(0.02, 0.15)
                aspect = random.uniform(0.3, 3.3)
                eh = int(round((target_area * aspect) ** 0.5))
                ew = int(round((target_area / aspect) ** 0.5))
                if 0 < eh < H and 0 < ew < W:
                    top = random.randint(0, H - eh)
                    left = random.randint(0, W - ew)
                    x[idx, :, top:top + eh, left:left + ew] = 0.0
    return x.to(memory_format=torch.channels_last)

def img_filenames(df):
    if "file_name" in df.columns:
        return df["file_name"].astype(str).values
    return np.array([f"{i}.jpg" for i in df["id"].astype(str).values])

class FastGameDataset(Dataset):
    def __init__(self, df, img_dir, cpu_transform=None, target=None):
        self.paths = [str(Path(img_dir) / fn) for fn in img_filenames(df)]
        self.cpu_transform = cpu_transform
        self.y = df[target].astype(int).values if target else None
        self.native_size = list(CFG["native_size"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        try:
            img = decode_jpeg(read_file(p), mode=ImageReadMode.RGB)
        except Exception:
            with Image.open(p) as im:
                img = v2.functional.pil_to_tensor(im.convert("RGB"))
        if img.shape[-2:] != CFG["native_size"]:
            img = v2.functional.resize(img, self.native_size, antialias=True)
        if self.cpu_transform is not None:
            img = self.cpu_transform(img)
        return (img, int(self.y[i])) if self.y is not None else (img, 0)

def make_loader(df, img_dir, cpu_transform, shuffle, target=None):
    nw = CFG["num_workers"]
    return DataLoader(
        FastGameDataset(df, img_dir, cpu_transform, target),
        batch_size=CFG["batch_size"] if shuffle else CFG["val_batch_size"],
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=(CFG["device"] == "cuda"),
        drop_last=shuffle,
        persistent_workers=(nw > 0),
        prefetch_factor=(4 if nw > 0 else None),
    )

@torch.inference_mode()
def predict_proba(model, loader, tta=True, scales=(1.0, 1.15)):
    """Softmax probabilities; TTA = mean over {image, left-right flip} x {native, 1.15x resolution}."""
    model.eval()
    out = []
    use_cuda = (CFG["device"] == "cuda")
    for x_u8, _ in tqdm(loader, desc="Predict", leave=False):
        x = gpu_preprocess(x_u8, erasing=False)
        with torch.autocast(CFG["device"], dtype=AMP_DTYPE, enabled=use_cuda):
            p = torch.softmax(model(x), 1)
            if tta:
                ps = [p, torch.softmax(model(torch.flip(x, dims=[3])), 1)]
                for s in scales[1:]:
                    xs = F.interpolate(x, scale_factor=s, mode="bilinear", align_corners=False)
                    ps += [torch.softmax(model(xs), 1), torch.softmax(model(torch.flip(xs, dims=[3])), 1)]
                p = torch.stack(ps).mean(0)
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

def confusion_plot(y, pred, labels=None, title="Confusion matrix (row-normalised, out-of-fold)"):
    cm = confusion_matrix(y, pred, normalize="true")
    k = cm.shape[0]
    plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues", vmin=0, vmax=1); plt.colorbar()
    for i in range(k):
        for j in range(k):
            plt.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
    plt.xticks(range(k), labels or range(k), rotation=30); plt.yticks(range(k), labels or range(k))
    plt.xlabel("predicted"); plt.ylabel("true"); plt.title(title); plt.tight_layout(); plt.show()

## 3. Load data and stratified 5-fold split

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
TARGET = "class" if "class" in train.columns else "label"
TRAIN_IMG, TEST_IMG = DATA / "train", DATA / "test"

print(f"Train shape: {train.shape}\nTest shape:  {test.shape}\nmissing labels: {int(train[TARGET].isna().sum())}")

skf = StratifiedKFold(CFG["n_folds"], shuffle=True, random_state=SEED)
train["fold"] = -1
for fold, (_, va) in enumerate(skf.split(train, train[TARGET])):
    train.loc[train.index[va], "fold"] = fold
print(train.groupby("fold")[TARGET].value_counts().unstack().set_axis(range(CFG["n_folds"])))

## 4. Training (5 folds x 2 backbones) and OOF evaluation

In [ ]:
TOTAL_FOLDS = len(MODELS) * len(CFG["train_folds"])
UNITS_PER_FOLD = CFG["epochs"] + 1.5
TOTAL_UNITS = TOTAL_FOLDS * UNITS_PER_FOLD
RUN_T0 = time.time()
ACTIVE_T0 = None
SKIPPED_UNITS = 0.0

test_loader = make_loader(test, TEST_IMG, None, False)   # built once, reused by every fold

def fmt_progress(units_done):
    global ACTIVE_T0
    pct = min(100.0, 100.0 * units_done / TOTAL_UNITS)
    active_u = units_done - SKIPPED_UNITS
    if ACTIVE_T0 is not None and active_u > 0.3:
        rem_u = max(0.0, TOTAL_UNITS - units_done)
        eta_min = ((time.time() - ACTIVE_T0) / active_u * rem_u) / 60.0
        return f"[{pct:5.1f}% | ETA ~{eta_min:4.1f}m]"
    return f"[{pct:5.1f}%]"

def run_fold(fold, model_name, fold_global_idx):
    global ACTIVE_T0, SKIPPED_UNITS
    base_u = fold_global_idx * UNITS_PER_FOLD
    safe_name = f"{model_name.replace('.', '_')}_{AUG}_{CFG['img_size'][0]}x{CFG['img_size'][1]}"
    cache_npz = SAVE_DIR / f"{safe_name}_fold{fold}_preds.npz"
    cache_pt = SAVE_DIR / "models" / f"{safe_name}_fold{fold}_ema.pt"

    trn, val = train[train.fold != fold].reset_index(drop=True), train[train.fold == fold]
    val_idx, val = val.index.values, val.reset_index(drop=True)

    if RESUME and cache_npz.exists():
        d = np.load(cache_npz)
        SKIPPED_UNITS += UNITS_PER_FOLD
        print(f"{fmt_progress(base_u + UNITS_PER_FOLD)} resume: {model_name} fold {fold} loaded (macro-F1 {float(d['f1']):.5f})")
        return d["val_idx"], d["best_val"], d["pt"]

    if ACTIVE_T0 is None:
        ACTIVE_T0 = time.time()

    fold_t0 = time.time()
    use_cuda = (CFG["device"] == "cuda")
    cw = compute_class_weight("balanced", classes=np.arange(CFG["num_classes"]), y=trn[TARGET].astype(int))
    w = torch.tensor(np.power(cw, 0.5), dtype=torch.float, device=CFG["device"])
    tr_loader = make_loader(trn, TRAIN_IMG, train_cpu_transform, True, TARGET)
    va_loader = make_loader(val, TRAIN_IMG, None, False, TARGET)

    model = timm.create_model(model_name, pretrained=True, num_classes=CFG["num_classes"]).to(
        CFG["device"], memory_format=torch.channels_last
    )
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    lr = MODEL_LR.get(model_name, 2e-4)
    head = [p for n, p in model.named_parameters() if n.startswith(("head", "classifier", "fc"))]
    ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in ids]
    try:
        opt = torch.optim.AdamW(
            [{"params": body, "lr": lr}, {"params": head, "lr": lr * CFG["head_lr_mult"]}],
            weight_decay=CFG["weight_decay"],
            fused=use_cuda,
        )
    except Exception:
        opt = torch.optim.AdamW(
            [{"params": body, "lr": lr}, {"params": head, "lr": lr * CFG["head_lr_mult"]}],
            weight_decay=CFG["weight_decay"],
        )

    # EMA is created from the uncompiled model (clean state_dict keys)
    if hasattr(timm.utils, "ModelEmaV3"):
        ema = timm.utils.ModelEmaV3(model, decay=CFG["ema_decay"], use_warmup=False)
    else:
        ema = timm.utils.ModelEmaV2(model, decay=CFG["ema_decay"])

    # compile only the training model (fixed input shape -> compiled once)
    if CFG["use_compile"] and use_cuda and hasattr(torch, "compile"):
        try:
            train_model = torch.compile(model)
        except Exception as e:
            print(f"torch.compile skipped ({e}); using eager mode")
            train_model = model
    else:
        train_model = model

    total, warm = CFG["epochs"] * len(tr_loader), CFG["warmup_epochs"] * len(tr_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warm if s < warm else max(1e-3, 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm)))),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(use_cuda and not USE_BF16))

    best_f1, best_val, best_state, best_ep = -1.0, None, None, 0
    no_imp = 0
    patience = CFG["early_stop_patience"]
    min_delta = CFG["early_stop_min_delta"]
    min_ep = CFG["early_stop_min_epoch"]
    stopped_ep = CFG["epochs"]

    for ep in range(CFG["epochs"]):
        train_model.train()
        loss_sum = 0.0
        t0 = time.time()
        for x_u8, y in tqdm(tr_loader, desc=f"{model_name[:12]} F{fold} Ep{ep+1}", leave=False):
            x = gpu_preprocess(x_u8, erasing=True)
            y = y.to(CFG["device"], non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(CFG["device"], dtype=AMP_DTYPE, enabled=use_cuda):
                loss = crit(train_model(x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                opt.step()
            sched.step()
            ema.update(model)
            loss_sum += loss.item() * len(x)

        pv = predict_proba(ema.module, va_loader, tta=False)
        f1 = f1_score(val[TARGET].astype(int), pv.argmax(1), average="macro")

        imp_sig = f1 > (best_f1 + min_delta)
        if f1 > best_f1:
            best_f1, best_val, best_ep = f1, pv, ep + 1
            best_state = {
                k: (v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu().clone())
                for k, v in ema.module.state_dict().items()
            }
        if imp_sig:
            no_imp = 0
            tag = "best"
        else:
            no_imp += 1
            tag = f"no_imp={no_imp}/{patience}"

        u_now = base_u + (ep + 1)
        print(
            f"{fmt_progress(u_now)} {model_name[:12]} fold {fold} | ep {ep + 1:02d}/{CFG['epochs']} | "
            f"loss {loss_sum / (len(tr_loader) * CFG['batch_size']):.4f} | val F1 {f1:.5f} ({tag}) | {time.time() - t0:4.1f}s"
        )

        if (ep + 1) >= min_ep and no_imp >= patience:
            stopped_ep = ep + 1
            saved_eps = CFG["epochs"] - stopped_ep
            SKIPPED_UNITS += saved_eps
            print(f"early stop: fold {fold} at epoch {stopped_ep}/{CFG['epochs']} (best F1 {best_f1:.5f} at epoch {best_ep})")
            break

    # save the best EMA weights and reload them for the test TTA
    torch.save(best_state, cache_pt)
    ema.module.load_state_dict({
        k: (v.float() if v.is_floating_point() else v)
        for k, v in best_state.items()
    })

    pt = predict_proba(ema.module, test_loader, tta=True)
    np.savez(cache_npz, val_idx=val_idx, best_val=best_val, pt=pt, f1=np.array(best_f1))
    print(
        f"{fmt_progress(base_u + UNITS_PER_FOLD)} {model_name[:12]} fold {fold} done (best epoch {best_ep}/{stopped_ep}) | "
        f"best-EMA macro-F1: {best_f1:.5f} | {(time.time() - fold_t0)/60:.1f} min"
    )

    del train_model, model, ema, opt, sched, scaler, tr_loader, va_loader, best_state
    try:
        torch._dynamo.reset()
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()
    return val_idx, best_val, pt

def run_all():
    oof_m, pt_m = {}, {}
    g_idx = 0
    for mname in MODELS:
        oof = np.full((len(train), CFG["num_classes"]), np.nan)
        pt = np.zeros((len(test), CFG["num_classes"]))
        for fold in CFG["train_folds"]:
            seed_everything(SEED + fold)
            vi, pv, pte = run_fold(fold, mname, g_idx)
            oof[vi] = pv
            pt += pte / len(CFG["train_folds"])
            g_idx += 1
        oof_m[mname], pt_m[mname] = oof, pt
        d = ~np.isnan(oof[:, 0])
        print(f"\n>>> {mname}: OOF macro-F1 = {f1_score(y_true[d], oof[d].argmax(1), average='macro'):.5f}\n")
    return oof_m, pt_m

y_true = train[TARGET].astype(int).values
oof_m, pt_m = run_all()

# ---- blend: plain mean of probabilities across backbones ----
oof = np.mean([oof_m[m] for m in MODELS], 0)
pt = np.mean([pt_m[m] for m in MODELS], 0)
done = ~np.isnan(oof[:, 0])
print(f"\nBLEND OOF macro-F1 on {int(done.sum())} rows ({len(CFG['train_folds'])}/{CFG['n_folds']} folds): {f1_score(y_true[done], oof[done].argmax(1), average='macro'):.5f}")

# ---- stacking: multinomial logistic regression on log-probabilities (fit on train OOF only) ----
if done.all() and len(MODELS) > 1:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_predict
    feat = lambda d: np.hstack([np.log(np.clip(d[m], 1e-6, 1)) for m in MODELS])
    Xs, Xt = feat(oof_m), feat(pt_m)
    meta = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced")
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED + 99)
    sp = cross_val_predict(meta, Xs, y_true, cv=cv, method="predict_proba")
    f_blend = f1_score(y_true, oof.argmax(1), average="macro")
    f_stack = f1_score(y_true, sp.argmax(1), average="macro")
    print(f"mean-blend OOF {f_blend:.5f} | stacked (nested CV) {f_stack:.5f}")
    if f_stack > f_blend:
        meta.fit(Xs, y_true); pt = meta.predict_proba(Xt); oof = sp
        print("-> using the stacked model for test predictions")
    else:
        print("-> stacking does not beat the mean blend; keeping the mean blend")
else:
    print("stacking skipped (needs all 5 folds and >= 2 backbones)")

np.save(SAVE_DIR / "task4_oof_proba.npy", oof); np.save(SAVE_DIR / "task4_test_proba.npy", pt)

# ---- optional label-shift correction (uses unlabeled test probabilities; off by default) ----
prior_tr = np.bincount(y_true, minlength=CFG["num_classes"]) / len(y_true)
if PRIOR_EM:
    pt_em, pi_test = em_label_shift(pt, prior_tr)
    print("estimated test class shares (EM):", dict(zip(NAMES, pi_test.round(3))), "| train:", dict(zip(NAMES, prior_tr.round(3))))
    print(f"labels changed by the correction: {(pt_em.argmax(1) != pt.argmax(1)).sum()} / {len(pt)}")
    pt = pt_em
print(f"\ntotal time: {(time.time() - RUN_T0)/60:.1f} min")
print("\n" + classification_report(y_true[done], oof[done].argmax(1), labels=range(CFG["num_classes"]), target_names=NAMES, digits=4, zero_division=0))
confusion_plot(y_true[done], oof[done].argmax(1), NAMES)

## 5. Predict and save

In [ ]:

pred = pt.argmax(1)
share = pd.DataFrame({
    "train": pd.Series(y_true).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
    "test predicted": pd.Series(pred).value_counts(normalize=True).reindex(range(CFG["num_classes"]), fill_value=0),
}).set_axis(NAMES)
print(share.round(4) * 100)
share.plot(kind="bar", figsize=(7, 4), color=["steelblue", "tomato"], title="Class share: train vs predicted test")
plt.xticks(rotation=30); plt.show()

# id, pred, p0..p4
df_pred = pd.DataFrame({"id": test["id"].values, "pred": pred})
for k in range(CFG["num_classes"]):
    df_pred[f"p{k}"] = pt[:, k]
pred_path = SAVE_DIR / "task4_pred.csv"
df_pred.to_csv(pred_path, index=False)

# id, task4 (column for submission_final.csv)
df_sub = pd.DataFrame({"id": test["id"].values, "task4": pred})
sub_path = SAVE_DIR / "task4_only_submission.csv"
df_sub.to_csv(sub_path, index=False)

print("saved:")
print(f" - {pred_path} ({len(df_pred):,} rows)")
print(f" - {sub_path} ({len(df_sub):,} rows)")
print(df_sub.head(10))

if IN_COLAB:
    from google.colab import files
    files.download(str(sub_path))
    files.download(str(pred_path))